# ML-03 — Frame Your Lane as an ML Task

> **Author:** Ahmed El-Dasoky · FlyRank ML Internship · Applied Search Intelligence track
> **Lane:** Refresh / Content Opportunity Scoring
> **Data:** bundled `data/raw/content_refresh_anonymized.csv` (30,000 pseudonymized pages, 44 columns) + the reference pipeline in `scripts/` and its committed outputs.

Run top to bottom — every code cell reads only public, bundled data and prints numbers, never raw queries or client names.


# ML-03 — Frame Your Lane as an ML Task


## 1. My lane as an ML task (type)

**Ranking / scoring.** I score every content item and return a descending queue; the score is a probability-like number (refresh priority). I am not asking 'is it declining?' (classification) as the final product — the working artifact is an ordered list, so *scoring then ranking* is the honest task framing. The classification problem (declining or not) appears only as an ingredient of the score.

## 2. Target or proxy

**Target (proxy):** `is_declining_label` = `trend_direction == 'down'`.
That label comes from the data's own 90-day trend evaluation (observed outcome shipped in the release), not from my own rule. I treat it as a proxy for 'this page needs attention' and pair it with *demand* (impressions/search_volume) so I do not flag healthy-but-small pages. No future window is used to build it. (Verified in the code cell below with the same definition the reference pipeline uses.)

## 3. Success metric

**Precision@50** on a client-holdout split — of the top 50 pages the queue recommends for review, the share that actually have the declining label (`trend_direction == 'down'`).
Why: a reviewer can only read ~50 pages per pass; what matters is how many of the first 50 are genuinely worth a human's time. Secondary metrics I still report: ROC-AUC, average precision, recall, and a baseline comparison on the *same split*.

## 4. The unit of analysis, as a real dataframe

One row = one content page (one client × one content item). Loading the lane slice and confirming the grain below:

In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
# one row per content_id·client_id
dups = df.duplicated(["content_id", "client_id"]).sum()
print("rows:", len(df), "· duplicate content_id×client_id:", dups)
print("distinct clients:", df["client_id"].nunique(), "· distinct content:", df["content_id"].nunique())

df["is_declining_label"] = df["trend_direction"].eq("down").astype(int)
print(df[["content_id","client_id","impressions_90d","avg_position","ctr","trend_direction","is_declining_label"]].head(3).to_string(index=False))

## 5. Why ML beats a fixed rule here

A hand rule like 'refresh pages with position > 10 and impressions > 1000' misses the interactions: a page can be `down` because demand collapsed (nothing to fix) or because engagement lags while impressions hold (a real fix). Combining deman, position, CTR, engagement, freshness, and trend non-linearly is exactly where a small scored model beats an if-statement — but the baseline in ML-07 still starts with the interpretable rule, because it is the honest yardstick.

## Self-check

- [x] Task type stated
- [x] Label defined and sourced
- [x] One metric with a reason
- [x] Grain verified on a real dataframe
- [x] Why ML (and why a rule first)